In [ ]:
#!/usr/bin/env python3
# combine_crop_ocr.py  (v2: keeps source-page mapping)

"""
Requirements (once):
    pip install pymupdf opencv-python-headless pytesseract numpy
    # plus tesseract-ocr installed system-wide
"""

from pathlib import Path
import fitz, cv2, numpy as np, pytesseract, re
from collections import defaultdict

# ───── configuration ──────────────────────────────────────────────────
IN_DIR   = Path("../data/mini")
OUT_PDF  = Path("../case_outputs/a.pdf")
CROP_PX  = (40, 20, 540, 70)       # (x0, y0, x1, y1) in pixels
CROP_DPI = 144                     # dpi for cropping pass
OCR_DPI  = 300                     # dpi for OCR pass
WORD     = "Benton"
RECURSE  = True                    # search sub-folders for PDFs?
# ──────────────────────────────────────────────────────────────────────


def concat_and_crop_bitmap(in_dir: Path, out_file: Path,
                           crop_px, crop_dpi=144, recurse=True):
    """
    Build a cropped-only PDF and return:
        • total pages written
        • mapping list: index i ↦ (Path pdf_path, int src_page_num)
    """
    x0, y0, x1, y1 = crop_px
    w_px, h_px   = x1 - x0, y1 - y0
    w_pt, h_pt   = w_px * 72 / crop_dpi, h_px * 72 / crop_dpi
    clip_rect    = fitz.Rect(x0 * 72 / crop_dpi,
                             y0 * 72 / crop_dpi,
                             x1 * 72 / crop_dpi,
                             y1 * 72 / crop_dpi)

    out_doc      = fitz.open()
    index_map    = []                            # output-page → (source-file, source-page)
    finder       = in_dir.rglob if recurse else in_dir.glob
    pdf_files    = sorted(finder("*.pdf")) + sorted(finder("*.PDF"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in {in_dir}")

    for pdf in pdf_files:
        with fitz.open(pdf) as src:
            for pg in src:
                pix = pg.get_pixmap(dpi=crop_dpi,
                                    clip=clip_rect,
                                    colorspace=fitz.csRGB,
                                    alpha=False)
                dst = out_doc.new_page(width=w_pt, height=h_pt)
                dst.insert_image(fitz.Rect(0, 0, w_pt, h_pt),
                                 stream=pix.tobytes("png"))
                # record where this output page came from
                index_map.append((pdf, pg.number + 1))  # pg.number is 0-based
    if not index_map:
        out_doc.close()
        raise ValueError("No pages processed; nothing to save")

    out_doc.save(out_file, garbage=4, deflate=True)
    out_doc.close()
    return len(index_map), index_map


def find_word_locations(out_pdf: Path, word: str, index_map, dpi=300):
    """
    OCR each cropped page; return dict{Path → [page_nums]} for pages containing *word*.
    """
    regex = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    hits  = defaultdict(list)

    with fitz.open(out_pdf) as doc:
        for out_idx, page in enumerate(doc):           # out_idx matches index_map
            pix = page.get_pixmap(dpi=dpi, colorspace=fitz.csRGB, alpha=False)
            img = np.frombuffer(pix.samples, np.uint8).reshape(
                      pix.height, pix.width, pix.n)    # height × width × 3 (RGB)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            thr  = cv2.threshold(gray, 0, 255,
                                 cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            if regex.search(pytesseract.image_to_string(thr)):
                src_path, src_page = index_map[out_idx]
                hits[src_path].append(src_page)
    return hits


if __name__ == "__main__":
    pages_written, index_map = concat_and_crop_bitmap(
        IN_DIR, OUT_PDF, crop_px=CROP_PX, crop_dpi=CROP_DPI, recurse=RECURSE)

    print(f"✓ Cropped & concatenated {pages_written} pages → {OUT_PDF}")

    occurrences = find_word_locations(OUT_PDF, WORD, index_map, dpi=OCR_DPI)
    if not occurrences:
        print(f'"{WORD}" not found in any source file.')
    else:
        print(f'\nPages containing "{WORD}":')
        for pdf_path in sorted(occurrences):
            pages = ", ".join(map(str, sorted(occurrences[pdf_path])))
            print(f"  • {pdf_path}  →  page(s): {pages}")
